# Causeway duration forecasting (canonical `travel_times`)

This notebook loads the **canonical** BigQuery table `swiftborder.causeway.travel_times` (live download with local CSV cache), then runs:

1. **XGBoost** (sklearn) — 60-minute horizon, **`jb_to_woodlands` only (JB → SG)** — SG → JB not implemented
2. **LSTM** (optional) — same tensors via [`timeseries_lstm.py`](../timeseries_lstm.py); plug in a **custom recurrent layer** when ready

Production serve remains **30-minute BQML** (`v_forecast_recent`). Score that with [`layer_b.py`](../layer_b.py), not this notebook.


## 1. Canonical dataset (BigQuery + cache)

Set `REFRESH_FROM_BQ = True` to re-pull the full table. Otherwise the notebook reuses `../data/causeway_gdata.csv`.


In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

import timeseries_xgb as tsx
from timeseries_xgb import TimeSeriesConfig

PROJECT = "swiftborder"
CONFIG = TimeSeriesConfig()
CACHE_PATH = Path("../data/causeway_gdata.csv")
REFRESH_FROM_BQ = False  # True = live download of full travel_times

export = tsx.sync_canonical_travel_times(CACHE_PATH, project=PROJECT, refresh=REFRESH_FROM_BQ)
raw = tsx.prepare_route_frame(export, CONFIG)
print("canonical rows:", len(export), "| route rows:", len(raw), "| route:", CONFIG.route_id)
print("cache:", CACHE_PATH.resolve())


## 2. XGBoost (60-minute horizon)

Pipeline matches teammate notebook `Test_Time_Series_Prediction` (2026-09-26): **300 s slew-rate cap** on `duration_in_traffic_sec`, **12 lag features**, tuned XGB (`n_estimators=200`, `max_depth=4`, `min_child_weight=50`, `subsample=0.6`). **`use_dwt=False`** (wavelet features optional; marginal on this history — see `TimeSeriesConfig.use_dwt` in [`timeseries_xgb.py`](../timeseries_xgb.py)).


In [ ]:
features = tsx.engineer_features(raw, CONFIG)
X_train, X_test, y_train, y_test = tsx.build_supervised_matrices(features, CONFIG)
xgb_model = tsx.train_xgb(X_train, y_train, config=CONFIG)
pred = xgb_model.predict(X_test)
print("XGB hold-out RMSE (min):", round(tsx.rmse_minutes(y_test, pred), 2))


In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

y_series, free_flow = tsx.regularized_series(raw, CONFIG)
dates = ["2026-09-22", "2026-09-23", "2026-09-24"]
scores = tsx.score_forecast_days(xgb_model, y_series, free_flow, dates, CONFIG)
print("RMSE (minutes)")
print(scores.pivot(index="method", columns="date", values="RMSE_min").round(2))

fig, axes = plt.subplots(1, 3, figsize=(24, 6), sharey=True)
for ax, d in zip(axes, dates):
    day_idx = pd.date_range(pd.Timestamp(d), periods=288, freq="5min")
    actual = y_series.reindex(day_idx) / 60
    ax.plot(actual.index, actual, color="black", lw=1.8, label="Actual")
    ax.set_title(d)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Minutes")
fig.suptitle(f"JB to Woodlands — {CONFIG.horizon_minutes} min ahead (XGB)")
plt.tight_layout()
plt.show()


## 3. LSTM (architectures + tuning)

Requires `pip install -r ../requirements-notebook.txt`. Same **JB→SG** `features` as XGB. Built-in designs: `lstm`, `stacked_lstm`, `bilstm`, `gru`, `residual_gated`, `bilstm_attention` — see [`timeseries_lstm.py`](../timeseries_lstm.py). CLI: `python train_lstm.py --mode compare|tune|train`.


In [ ]:
RUN_LSTM = False  # True after tensorflow install
LSTM_MODE = "train"  # train | compare | tune

if RUN_LSTM:
    import timeseries_lstm as tsl

    if LSTM_MODE == "train":
        cfg = tsl.LSTMTrainConfig(architecture="residual_gated", lstm_units=64, epochs=25)
        model, result = tsl.train_lstm(features, CONFIG, cfg)
        print("LSTM:", result.metrics, "| persistence:", result.persistence_metrics)
    elif LSTM_MODE == "compare":
        print(tsl.compare_lstm_architectures(features, CONFIG, base_config=tsl.LSTMTrainConfig(epochs=20)))
    else:
        tune = tsl.tune_lstm_hyperparameters(features, CONFIG, max_trials=12)
        print("best:", tune.best_config, tune.best_metrics)
else:
    print("LSTM skipped. Set RUN_LSTM=True.")
